# DarijaCare — ré-entraînement du classifieur Laya (Kaggle / Colab, GPU T4)

Protocole de la fiche `mballouch/jev-darija-telecom-model` (v5), appliqué à **notre** découpage
train / validation (graine 42) : départ `mballouch/jev-darija`, 22 blocs d'encodeur + couches de
décision entraînés (125 M paramètres), embeddings gelés, entropie croisée avec lissage 0,05,
ordre des options mélangé, AdamW (3e-5 / 1e-4), échauffement 6 % puis cosinus, batch effectif 32,
fp16, meilleure époque sur la validation, températures de confiance ajustées sur la validation.

**Avant de lancer** : *Settings → Accelerator → GPU T4*, *Internet → On*.
Une session Kaggle dure au plus 12 h : l'entraînement **reprend automatiquement** à la dernière
époque terminée si vous relancez le notebook avec le dossier de sortie conservé
(*Save Version → Save output* puis ajout de la sortie comme *Input*).

Données synthétiques (CC BY 4.0, Mohamed Ballouch). Modèle de base Apache-2.0.

## 1. Paramètres

In [ ]:
REPO_URL = ""          # URL git du dépôt DarijaCare ; vide = code fourni comme dataset Kaggle
SRC_DATASET = "/kaggle/input/darijacare-src"   # utilisé si REPO_URL est vide
SEEDS = [42]           # une graine par session conseillée ; puis 43, 44
WORK = "/kaggle/working/darijacare"
RESUME_FROM = ""       # dossier models/retrain d'une session précédente (Input Kaggle), sinon vide
MICRO_BATCH = 8        # baisser à 4 (et GRAD_ACCUM à 8) en cas de manque de mémoire
GRAD_ACCUM = 4

## 2. Code et dépendances (versions figées, torch de l'environnement conservé)

In [ ]:
import os, shutil, subprocess, sys
if REPO_URL:
    if not os.path.exists(WORK):
        subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
else:
    if not os.path.exists(WORK):
        shutil.copytree(SRC_DATASET, WORK)
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "notebooks/requirements-kaggle.txt"], check=True)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
os.environ["PYTHONUTF8"] = "1"
sys.path.insert(0, os.path.join(WORK, "src"))
import torch, transformers, laya
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| laya", getattr(laya, "__version__", "?"), "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN")
assert torch.cuda.is_available(), "Activer un GPU (T4) dans les paramètres du notebook"

## 3. Données : reconstruction depuis le Hub (révision figée) et contrôle des empreintes

In [ ]:
import hashlib, json
subprocess.run([sys.executable, "-m", "darijacare.data.cli", "all"], check=True)
expected = json.load(open("reports/split/report.json", encoding="utf-8"))["sha256"] if False else None

In [ ]:
# Les empreintes attendues sont celles du rapport versionné dans git (avant reconstruction).
expected = json.loads(subprocess.run(["git", "show", "HEAD:reports/split/report.json"], capture_output=True,
                                     text=True).stdout or "{}").get("sha256") if REPO_URL else None
if expected is None:
    expected = {"train.jsonl": "6b30dc2ec90d86edf770f689f4df654a394b13c2a5b952e5790781cdbbd10f97",
                "val.jsonl": "046de1e4b14f63064a2686586dd444fe9d725864df42743d154a21147b52123c",
                "test.jsonl": "82df7427ef4d8bd7b8c107f4200a1e0753c2ddf7bb454978da0c46d006094a77"}
for name, sha in expected.items():
    got = hashlib.sha256(open(f"data/processed/{name}", "rb").read()).hexdigest()
    print(name, "OK" if got == sha else f"DIFFÉRENT ({got})")
    assert got == sha, "Le découpage diffère de celui du dépôt : ne pas entraîner."

## 4. Point de départ : `mballouch/jev-darija` (révision figée)

In [ ]:
from huggingface_hub import snapshot_download
REV = "bf18d80a39d03176849e68d0510a87112d321d2a"
path = snapshot_download("mballouch/jev-darija", revision=REV, local_dir="models/jev-darija")
sha = hashlib.sha256(open("models/jev-darija/model.safetensors", "rb").read()).hexdigest()
print(path, sha)
assert sha == "54c7defae78a9b2124f926e3971527018c6bde13f40ae50e050fae313a11db22"

## 5. Entraînement (reprise automatique) — une graine par session

In [ ]:
if RESUME_FROM and not os.path.exists("models/retrain"):
    shutil.copytree(RESUME_FROM, "models/retrain")
cmd = [sys.executable, "-m", "darijacare.nlu.train", "--seeds", *map(str, SEEDS),
       "--micro-batch", str(MICRO_BATCH), "--grad-accum", str(GRAD_ACCUM), "--out", "models/retrain"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 6. Résultats de validation

Le test officiel n'est **pas** utilisé ici : l'évaluation finale se fait une seule fois, sur le
modèle retenu, avec `python -m darijacare.nlu.cli evaluate --model retrain-seed42 --checkpoint models/retrain/seed42/best --split test`.

In [ ]:
for s in SEEDS:
    summ = json.load(open(f"models/retrain/seed{s}/summary.json", encoding="utf-8"))
    print("graine", s, "meilleure époque", summ["best"], "températures", summ["temperature"])
    for h in summ["history"]:
        print("   ", h)

## 7. Archive à télécharger (meilleur checkpoint + historique, sans l'état d'optimiseur)

In [ ]:
for s in SEEDS:
    pack = f"/kaggle/working/pack-seed{s}"
    shutil.rmtree(pack, ignore_errors=True)
    shutil.copytree(f"models/retrain/seed{s}/best", f"{pack}/best")
    shutil.copy(f"models/retrain/seed{s}/summary.json", pack)
    print(shutil.make_archive(f"/kaggle/working/darijacare-retrain-seed{s}", "zip", pack))